# 03 — Time Series Models, Ensemble, Calibration, Prediction

Uses forecast-like validation and raw-derived recovery anchoring. Export is disabled by default.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def find_package_paths():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        # Final GitHub layout: repo_root/Raw Data and repo_root/part3_forecasting/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "part3_forecasting" / "notebooks").exists():
            return candidate, candidate / "part3_forecasting"
        # Backward-compatible layout used during development: package_root/Raw Data and package_root/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "notebooks").exists():
            return candidate, candidate
    raise FileNotFoundError("Cannot locate repository root with Raw Data and forecasting notebooks")

REPO_ROOT, PACKAGE_ROOT = find_package_paths()
RAW_DIR = REPO_ROOT / "Raw Data"
NB_DIR = PACKAGE_ROOT / "notebooks"
ARTIFACT_DIR = PACKAGE_ROOT / "artifacts"
REPORT_DIR = PACKAGE_ROOT / "reports"
OUTPUT_DIR = PACKAGE_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RAW_DIR =", RAW_DIR)


## Load Features and Model Libraries

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.optimize import minimize
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None
train = pd.read_parquet(ARTIFACT_DIR / "train_features.parquet")
test = pd.read_parquet(ARTIFACT_DIR / "test_features.parquet")
feature_cols = json.loads((ARTIFACT_DIR / "feature_cols.json").read_text(encoding="utf-8"))
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
feature_cols = [c for c in feature_cols if c in train.columns and c in test.columns]
print(train.shape, test.shape, len(feature_cols))


## Forecast-Like Helpers

In [ ]:
def metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

def target_history_cols(cols):
    return [c for c in cols if c.startswith("lag_Revenue_") or c.startswith("roll_Revenue_") or c.startswith("lag_COGS_") or c.startswith("roll_COGS_")]

def sanitize_future_features(history, future, cols):
    # For validation/test rows, overwrite target-history columns with profiles fit only on available history.
    out = future.copy()
    h = history.copy()
    h["month_day"] = h["Date"].dt.strftime("%m-%d")
    h["month"] = h["Date"].dt.month
    out["month_day_tmp"] = out["Date"].dt.strftime("%m-%d")
    for col in target_history_cols(cols):
        if col not in out.columns or col not in h.columns:
            continue
        md = h.groupby("month_day")[col].median()
        mo = h.groupby("month")[col].median()
        global_val = h[col].median()
        out[col] = out["month_day_tmp"].map(md).fillna(out["month"].map(mo)).fillna(global_val).fillna(0)
    return out.drop(columns=["month_day_tmp"], errors="ignore")

def derive_recovery_anchor(history, target="Revenue"):
    h = history.copy()
    h["year"] = h["Date"].dt.year
    pre = h[h["year"].between(2016, 2018)][target]
    if pre.empty:
        pre = h[h["year"] <= min(2018, h["year"].max())][target]
    latest_year = int(h["year"].max())
    latest = h[h["year"].eq(latest_year)][target]
    if latest.empty:
        latest = h[target].tail(365)
    pre_mean = float(pre.mean())
    latest_mean = float(latest.mean())
    anchor = 0.5 * pre_mean + 0.5 * latest_mean
    return anchor, {"pre_maturity_mean": pre_mean, "latest_year": latest_year, "latest_mean": latest_mean, "anchor": anchor}

def seasonal_recovery_forecast(history, pred_dates, target="Revenue"):
    h = history[["Date", target]].dropna().copy()
    h["year"] = h["Date"].dt.year
    h["month"] = h["Date"].dt.month
    h["day_of_week"] = h["Date"].dt.dayofweek
    h["month_day"] = h["Date"].dt.strftime("%m-%d")
    shape_source = h[h["year"].between(2013, 2018)]
    if shape_source.empty:
        shape_source = h
    daily_ratio = shape_source.groupby("month_day")[target].mean() / shape_source[target].mean()
    month_ratio = shape_source.groupby("month")[target].mean() / shape_source[target].mean()
    dow_ratio = shape_source.groupby("day_of_week")[target].mean() / shape_source[target].mean()
    anchor, info = derive_recovery_anchor(history, target)
    pred = pd.DataFrame({"Date": pd.to_datetime(pred_dates)})
    pred["month"] = pred["Date"].dt.month
    pred["day_of_week"] = pred["Date"].dt.dayofweek
    pred["month_day"] = pred["Date"].dt.strftime("%m-%d")
    ratio = pred["month_day"].map(daily_ratio).fillna(pred["month"].map(month_ratio)).fillna(1.0)
    ratio = 0.85 * ratio + 0.15 * pred["day_of_week"].map(dow_ratio).fillna(1.0)
    values = ratio.to_numpy(dtype=float)
    values = values / np.nanmean(values) * anchor
    return np.clip(values, 0, None), info

def fit_predict_models(history, future, target):
    cols = [c for c in feature_cols if c in history.columns and c in future.columns]
    Xh, Xf = history[cols], future[cols]
    y = history[target].astype(float)
    preds = {}
    ridge = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("ridge", Ridge(alpha=20.0))])
    ridge.fit(Xh, np.log1p(y))
    preds["ridge_log"] = np.expm1(ridge.predict(Xf)).clip(0)
    if LGBMRegressor is not None:
        lgbm = LGBMRegressor(objective="mae", n_estimators=900, learning_rate=0.025, num_leaves=31, min_child_samples=30, subsample=0.85, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)
        lgbm.fit(Xh, y)
        preds["lgbm_l1"] = np.clip(lgbm.predict(Xf), 0, None)
    if XGBRegressor is not None:
        xgb = XGBRegressor(objective="reg:absoluteerror", n_estimators=800, learning_rate=0.025, max_depth=5, min_child_weight=20, subsample=0.85, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1, verbosity=0, tree_method="hist")
        xgb.fit(Xh, y)
        preds["xgb_l1"] = np.clip(xgb.predict(Xf), 0, None)
    if CatBoostRegressor is not None:
        cat = CatBoostRegressor(loss_function="MAE", iterations=800, learning_rate=0.03, depth=6, random_seed=RANDOM_SEED, verbose=False, allow_writing_files=False)
        cat.fit(Xh, y)
        preds["catboost_l1"] = np.clip(cat.predict(Xf), 0, None)
    return preds

def inverse_mae_weights(oof, target):
    model_mae = oof[oof["target"].eq(target)].groupby("model").apply(lambda g: mean_absolute_error(g["actual"], g["prediction"]))
    inv = 1 / model_mae.clip(lower=1)
    w = (inv / inv.sum()).to_dict()
    return {k: float(v) for k, v in w.items()}

def _oof_wide(oof, target):
    target_oof = oof[oof["target"].eq(target)].copy()
    if target_oof.empty:
        return pd.DataFrame()
    wide = target_oof.pivot_table(
        index=["Date", "fold", "actual"],
        columns="model",
        values="prediction",
        aggfunc="mean",
    ).reset_index()
    model_cols = [c for c in wide.columns if c not in ["Date", "fold", "actual"]]
    for col in model_cols:
        wide[col] = wide[col].fillna(wide[col].mean())
    if model_cols:
        wide[model_cols] = wide[model_cols].T.fillna(wide[model_cols].mean(axis=1)).T
        wide[model_cols] = wide[model_cols].fillna(wide["actual"].mean())
    return wide

def fit_meta_learner(oof: pd.DataFrame, target: str) -> dict[str, float]:
    fallback = inverse_mae_weights(oof, target)
    wide = _oof_wide(oof, target)
    model_cols = [c for c in wide.columns if c not in ["Date", "fold", "actual"]]
    debug = {
        "target": target,
        "method": "lasso_positive",
        "meta_r2": np.nan,
        "fallback_used": False,
        "fallback_reason": "",
    }
    if len(model_cols) < 2 or wide.empty:
        debug.update({"method": "inverse_mae", "fallback_used": True, "fallback_reason": "<2 models or empty OOF"})
        META_DEBUG_ROWS.append(debug | {f"weight_{k}": v for k, v in fallback.items()})
        print(f"Meta-learner fallback for {target}: {debug['fallback_reason']}")
        return fallback
    X = wide[model_cols].astype(float)
    y = wide["actual"].astype(float)
    try:
        cv = min(5, len(wide))
        meta = LassoCV(cv=cv, positive=True, max_iter=5000, random_state=RANDOM_SEED)
        meta.fit(X, y)
        pred = meta.predict(X)
        meta_r2 = float(r2_score(y, pred))
        coefs = pd.Series(meta.coef_, index=model_cols, dtype=float).clip(lower=0)
        if meta_r2 < 0 or coefs.sum() <= 0:
            reason = f"bad meta fit: R2={meta_r2:.4f}, coef_sum={coefs.sum():.6f}"
            debug.update({"method": "inverse_mae", "meta_r2": meta_r2, "fallback_used": True, "fallback_reason": reason})
            META_DEBUG_ROWS.append(debug | {f"weight_{k}": v for k, v in fallback.items()})
            print(f"Meta-learner fallback for {target}: {reason}")
            return fallback
        weights = (coefs / coefs.sum()).to_dict()
        debug.update({"meta_r2": meta_r2})
        META_DEBUG_ROWS.append(debug | {f"weight_{k}": float(v) for k, v in weights.items()})
        print(f"Meta-learner weights for {target}:")
        for name, weight in sorted(weights.items(), key=lambda kv: kv[1], reverse=True):
            print(f"  {name:<24} {weight:.4f}")
        print(f"  Meta R2 (OOF):       {meta_r2:.3f}")
        return {k: float(v) for k, v in weights.items()}
    except Exception as exc:
        reason = repr(exc)
        debug.update({"method": "inverse_mae", "fallback_used": True, "fallback_reason": reason})
        META_DEBUG_ROWS.append(debug | {f"weight_{k}": v for k, v in fallback.items()})
        print(f"Meta-learner fallback for {target}: {reason}")
        return fallback

def ensemble_oof_by_weights(oof, target, weights):
    wide = _oof_wide(oof, target)
    model_cols = [c for c in wide.columns if c not in ["Date", "fold", "actual"]]
    active = [c for c in model_cols if weights.get(c, 0) > 0]
    if not active:
        return pd.DataFrame()
    total = sum(weights[c] for c in active)
    pred = np.zeros(len(wide), dtype=float)
    for col in active:
        pred += (weights[col] / total) * wide[col].astype(float).to_numpy()
    out = wide[["Date", "fold", "actual"]].copy()
    out["prediction"] = pred
    out["target"] = target
    return out

META_DEBUG_ROWS = []

FOLDS = [
    ("Fold_2017", "2016-12-31", "2017-01-01", "2017-12-31"),
    ("Fold_2018", "2017-12-31", "2018-01-01", "2018-12-31"),
    ("Fold_2021", "2020-12-31", "2021-01-01", "2021-12-31"),
    ("Fold_2022", "2021-12-31", "2022-01-01", "2022-12-31"),
    ("Fold_H548_2021_2022", "2021-06-30", "2021-07-01", "2022-12-30"),
]


## Rolling-Origin OOF

In [ ]:
oof_parts = []
metric_rows = []
for fold, train_end, val_start, val_end in FOLDS:
    hist = train[train["Date"] <= train_end].copy()
    val_raw = train[train["Date"].between(val_start, val_end)].copy()
    if len(hist) < 365 or val_raw.empty:
        continue
    val = sanitize_future_features(hist, val_raw, feature_cols)
    for target in ["Revenue", "COGS"]:
        seasonal_pred, anchor_info = seasonal_recovery_forecast(hist, val["Date"], target)
        oof_parts.append(pd.DataFrame({"Date": val["Date"], "fold": fold, "model": "raw_recovery_seasonal", "target": target, "actual": val_raw[target], "prediction": seasonal_pred}))
        metric_rows.append({"fold": fold, "model": "raw_recovery_seasonal", "target": target, **metrics(val_raw[target], seasonal_pred), **anchor_info})
        for model_name, pred in fit_predict_models(hist, val, target).items():
            oof_parts.append(pd.DataFrame({"Date": val["Date"], "fold": fold, "model": model_name, "target": target, "actual": val_raw[target], "prediction": pred}))
            metric_rows.append({"fold": fold, "model": model_name, "target": target, **metrics(val_raw[target], pred)})
oof = pd.concat(oof_parts, ignore_index=True)
metrics_df = pd.DataFrame(metric_rows).sort_values(["target", "MAE"])
oof.to_parquet(ARTIFACT_DIR / "03_model_oof.parquet", index=False)
metrics_df.to_csv(REPORT_DIR / "03_model_metrics.csv", index=False)
metrics_df.head(30)


## Sale Event Feature Validation


In [ ]:
SALE_EVENT_FEATURES = [
    "is_1111", "is_1212", "is_828",
    "days_to_1111", "1111_pre_7", "1111_pre_3", "1111_day", "1111_post_3",
    "days_to_1212", "1212_pre_7", "1212_pre_3", "1212_day", "1212_post_3",
]

sale_event_report = []
sale_event_importance = []
sale_cols = [c for c in SALE_EVENT_FEATURES if c in feature_cols and c in train.columns]
if LGBMRegressor is None:
    print("Sale event validation skipped: LightGBM is not installed.")
elif not sale_cols:
    print("Sale event validation skipped: sale event features not found. Run notebook 01 before rerunning notebook 03.")
else:
    hist = train[train["Date"] <= "2021-12-31"].copy()
    val_raw = train[train["Date"].between("2022-01-01", "2022-12-31")].copy()
    val = sanitize_future_features(hist, val_raw.copy(), feature_cols)
    base_cols = [
        c for c in feature_cols
        if c in hist.columns and c in val.columns and c not in sale_cols and pd.api.types.is_numeric_dtype(hist[c])
    ]
    after_cols = [
        c for c in feature_cols
        if c in hist.columns and c in val.columns and pd.api.types.is_numeric_dtype(hist[c])
    ]
    imputer_base = SimpleImputer(strategy="median")
    X_base = imputer_base.fit_transform(hist[base_cols])
    X_base_val = imputer_base.transform(val[base_cols])
    imputer_after = SimpleImputer(strategy="median")
    X_after = imputer_after.fit_transform(hist[after_cols])
    X_after_val = imputer_after.transform(val[after_cols])
    y = hist["Revenue"].astype(float)
    y_val = val_raw["Revenue"].astype(float)
    base_model = LGBMRegressor(objective="mae", n_estimators=900, learning_rate=0.025, num_leaves=31, min_child_samples=30, subsample=0.85, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)
    after_model = LGBMRegressor(objective="mae", n_estimators=900, learning_rate=0.025, num_leaves=31, min_child_samples=30, subsample=0.85, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)
    base_model.fit(X_base, y)
    after_model.fit(X_after, y)
    base_pred = np.clip(base_model.predict(X_base_val), 0, None)
    after_pred = np.clip(after_model.predict(X_after_val), 0, None)
    before_mae = mean_absolute_error(y_val, base_pred)
    after_mae = mean_absolute_error(y_val, after_pred)
    sale_event_report.append({
        "fold": "Fold_2022",
        "target": "Revenue",
        "model": "lgbm_l1",
        "before_features": len(base_cols),
        "after_features": len(after_cols),
        "sale_event_features": len(sale_cols),
        "MAE_before": float(before_mae),
        "MAE_after": float(after_mae),
        "delta": float(after_mae - before_mae),
        "delta_pct": float((after_mae - before_mae) / before_mae) if before_mae else np.nan,
    })
    importance = pd.Series(after_model.feature_importances_, index=after_cols)
    sale_event_importance = (
        importance.reindex(sale_cols)
        .fillna(0)
        .rename_axis("feature")
        .reset_index(name="importance")
        .sort_values("importance", ascending=False)
    )
    display(pd.DataFrame(sale_event_report))
    display(sale_event_importance)

pd.DataFrame(sale_event_report).to_csv(REPORT_DIR / "03_sale_event_lgbm_fold2022.csv", index=False)
if isinstance(sale_event_importance, pd.DataFrame):
    sale_event_importance.to_csv(REPORT_DIR / "03_sale_event_feature_importance.csv", index=False)
else:
    pd.DataFrame(columns=["feature", "importance"]).to_csv(REPORT_DIR / "03_sale_event_feature_importance.csv", index=False)


## Meta-Learner Fold_2022 Comparison


In [ ]:
inverse_weights = {target: inverse_mae_weights(oof, target) for target in ["Revenue", "COGS"]}
meta_weights = {target: fit_meta_learner(oof, target) for target in ["Revenue", "COGS"]}
pd.DataFrame(META_DEBUG_ROWS).to_csv(REPORT_DIR / "03_meta_learner_debug.csv", index=False)

comparison_rows = []
for target in ["Revenue", "COGS"]:
    inv_pred = ensemble_oof_by_weights(oof, target, inverse_weights[target])
    meta_pred = ensemble_oof_by_weights(oof, target, meta_weights[target])
    for fold in sorted(set(inv_pred["fold"]).intersection(set(meta_pred["fold"]))):
        inv_fold = inv_pred[inv_pred["fold"].eq(fold)]
        meta_fold = meta_pred[meta_pred["fold"].eq(fold)]
        inv_mae = mean_absolute_error(inv_fold["actual"], inv_fold["prediction"])
        meta_mae = mean_absolute_error(meta_fold["actual"], meta_fold["prediction"])
        comparison_rows.append({
            "target": target,
            "fold": fold,
            "inverse_mae_MAE": float(inv_mae),
            "meta_learner_MAE": float(meta_mae),
            "delta": float(meta_mae - inv_mae),
            "delta_pct": float((meta_mae - inv_mae) / inv_mae) if inv_mae else np.nan,
        })
fold2022_comparison = pd.DataFrame(comparison_rows)
fold2022_comparison.to_csv(REPORT_DIR / "03_fold2022_ensemble_comparison.csv", index=False)
display(fold2022_comparison[fold2022_comparison["fold"].eq("Fold_2022")])


## Final Revenue Ensemble and Raw-Derived Anchor

In [ ]:
full_hist = train[train["Date"].between("2013-01-01", "2022-12-31")].copy()
test_future = sanitize_future_features(full_hist, test.copy(), feature_cols)

weights = inverse_weights
(REPORT_DIR / "03_ensemble_weights.json").write_text(json.dumps(weights, indent=2), encoding="utf-8")

revenue_model_preds = fit_predict_models(full_hist, test_future, "Revenue")
seasonal_revenue, revenue_anchor_info = seasonal_recovery_forecast(full_hist, test["Date"], "Revenue")
revenue_model_preds["raw_recovery_seasonal"] = seasonal_revenue
rev_weights = weights["Revenue"]
if "raw_recovery_seasonal" not in rev_weights:
    rev_weights["raw_recovery_seasonal"] = 0.35
    s = sum(rev_weights.values())
    rev_weights = {k: v / s for k, v in rev_weights.items()}

revenue_ml = np.zeros(len(test))
total = 0
for name, pred in revenue_model_preds.items():
    w = rev_weights.get(name, 0)
    if w > 0:
        revenue_ml += w * pred
        total += w
if total == 0:
    revenue_ml = seasonal_revenue.copy()
else:
    revenue_ml /= total

anchor_mean = revenue_anchor_info["anchor"]
revenue_ml = revenue_ml / np.mean(revenue_ml) * anchor_mean
seasonal_revenue = seasonal_revenue / np.mean(seasonal_revenue) * anchor_mean
final_revenue = 0.55 * seasonal_revenue + 0.45 * revenue_ml
final_revenue = np.clip(final_revenue, 0, None)
anchor_report = {"Revenue": revenue_anchor_info, "final_revenue_mean": float(np.mean(final_revenue))}
(REPORT_DIR / "03_final_anchor_summary.json").write_text(json.dumps(anchor_report, indent=2), encoding="utf-8")
pd.Series(final_revenue).describe()


## COGS Ratio Forecast

In [ ]:
ratio_train = full_hist.copy()
ratio_train["COGS_ratio_target"] = ratio_train["COGS"] / ratio_train["Revenue"].replace(0, np.nan)
ratio_train = ratio_train.replace([np.inf, -np.inf], np.nan).dropna(subset=["COGS_ratio_target"])
ratio_cols = [c for c in feature_cols if c in ratio_train.columns and c in test_future.columns]

ratio_model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("ridge", Ridge(alpha=10.0))])
ratio_model.fit(ratio_train[ratio_cols], ratio_train["COGS_ratio_target"])
ratio_pred_model = ratio_model.predict(test_future[ratio_cols])

ratio_train["month"] = ratio_train["Date"].dt.month
ratio_train["month_day"] = ratio_train["Date"].dt.strftime("%m-%d")
pre_ratio = ratio_train[ratio_train["Date"].dt.year.between(2013, 2018)]
if pre_ratio.empty:
    pre_ratio = ratio_train
md_ratio = pre_ratio.groupby("month_day")["COGS_ratio_target"].median()
mo_ratio = pre_ratio.groupby("month")["COGS_ratio_target"].median()
test_tmp = test[["Date"]].copy()
test_tmp["month"] = test_tmp["Date"].dt.month
test_tmp["month_day"] = test_tmp["Date"].dt.strftime("%m-%d")
ratio_pred_seasonal = test_tmp["month_day"].map(md_ratio).fillna(test_tmp["month"].map(mo_ratio)).fillna(ratio_train["COGS_ratio_target"].median()).to_numpy()

lo, hi = ratio_train["COGS_ratio_target"].quantile([0.02, 0.98])
final_ratio = np.clip(0.65 * ratio_pred_seasonal + 0.35 * ratio_pred_model, lo, hi)
final_cogs = final_revenue * final_ratio

cogs_report = {
    "ratio_low": float(lo),
    "ratio_high": float(hi),
    "ratio_mean": float(np.mean(final_ratio)),
    "method": "Revenue * clipped blended seasonal/ridge COGS ratio",
}
pd.DataFrame([cogs_report]).to_csv(REPORT_DIR / "03_cogs_ratio_selection.csv", index=False)
cogs_report


## Final Prediction DataFrame

In [ ]:
final = test[["Date"]].copy()
final["Revenue"] = final_revenue
final["COGS"] = final_cogs
sample_dates = pd.read_csv(RAW_DIR / "sample_submission.csv", usecols=["Date"], parse_dates=["Date"])
assert len(final) == 548
assert final["Date"].tolist() == sample_dates["Date"].tolist()
assert final[["Revenue", "COGS"]].notna().all().all()
assert (final[["Revenue", "COGS"]] >= 0).all().all()
summary = final.assign(month=final["Date"].dt.month).groupby("month")[["Revenue", "COGS"]].mean().reset_index()
summary.to_csv(REPORT_DIR / "03_final_monthly_shape.csv", index=False)
display(final[["Revenue", "COGS"]].describe())
display(summary)


## SHAP Explainability and Improvement Signals

This section trains lightweight explanation models after the final forecast is built. It explains:

- which feature families drive the Revenue forecast;
- which feature families drive the COGS/Revenue ratio forecast;
- whether the model relies more on seasonality, Tet, promo timing, fashion cycles, lag momentum, or raw-derived operational profiles.

Outputs are saved under `reports/` so you can inspect them after each run and decide the next modeling improvement.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    shap = None
    SHAP_AVAILABLE = False
    print("SHAP unavailable; falling back to native model importance:", repr(exc))


def feature_family(feature: str) -> str:
    f = feature.lower()
    if f.startswith("lag_revenue") or f.startswith("roll_revenue"):
        return "Revenue lag momentum"
    if f.startswith("lag_cogs") or f.startswith("roll_cogs") or "cogs_ratio" in f:
        return "COGS / margin history"
    if "tet" in f or f == "days_to_tet":
        return "Tet timing"
    if "black_friday" in f:
        return "Black Friday"
    if any(x in f for x in ["1111", "1212", "828", "sale_event"]):
        return "Mega sale events"
    if "double_day" in f or "date_eq_month" in f:
        return "Double-day commerce"
    if "payday" in f:
        return "Payday timing"
    if "promo" in f:
        return "Promotion intensity"
    if "fashion_micro_season" in f or any(x in f for x in ["spring_collection", "summer_sale", "fall_collection", "winter_clearance", "back_to_school", "yearend"]):
        return "Fashion seasonality"
    if f.startswith("profile_sessions") or f.startswith("profile_unique") or f.startswith("profile_page") or "traffic" in f:
        return "Traffic profile"
    if any(x in f for x in ["stock", "inventory", "supply", "fill_rate", "sell_through"]):
        return "Inventory profile"
    if any(x in f for x in ["return", "refund"]):
        return "Return/refund profile"
    if any(x in f for x in ["payment", "installments", "cod"]):
        return "Payment risk profile"
    if "category_share" in f:
        return "Category mix profile"
    if "is_vn_major_holiday" in f or any(x in f for x in ["month", "day", "week", "quarter", "sin_", "cos_", "weekend"]):
        return "Calendar seasonality"
    return "Other"


def imputed_xy(train_df, future_df, cols, target):
    cols = [c for c in cols if c in train_df.columns and c in future_df.columns and pd.api.types.is_numeric_dtype(train_df[c])]
    imputer = SimpleImputer(strategy="median")
    X_train = pd.DataFrame(imputer.fit_transform(train_df[cols]), columns=cols, index=train_df.index)
    X_future = pd.DataFrame(imputer.transform(future_df[cols]), columns=cols, index=future_df.index)
    y = train_df[target].astype(float).to_numpy()
    return X_train, X_future, y, cols


def fit_explain_model(X, y, seed=RANDOM_SEED):
    if LGBMRegressor is not None:
        model = LGBMRegressor(
            objective="mae",
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
        model.fit(X, y)
        return model, "lightgbm"
    if XGBRegressor is not None:
        model = XGBRegressor(
            objective="reg:absoluteerror",
            n_estimators=600,
            learning_rate=0.03,
            max_depth=5,
            min_child_weight=20,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
            tree_method="hist",
        )
        model.fit(X, y)
        return model, "xgboost"
    model = Ridge(alpha=20.0)
    model.fit(X, y)
    return model, "ridge"


def compute_importance(model, model_kind, X_reference, X_forecast, label):
    sample_n = min(1200, len(X_reference))
    ref_sample = X_reference.sample(sample_n, random_state=RANDOM_SEED) if len(X_reference) > sample_n else X_reference.copy()
    forecast_sample = X_forecast.copy()
    if len(forecast_sample) > 548:
        forecast_sample = forecast_sample.sample(548, random_state=RANDOM_SEED)

    source = "native_importance"
    ref_importance = None
    forecast_importance = None

    if SHAP_AVAILABLE and model_kind in {"lightgbm", "xgboost"}:
        try:
            explainer = shap.TreeExplainer(model)
            ref_values = explainer.shap_values(ref_sample)
            future_values = explainer.shap_values(forecast_sample)
            if isinstance(ref_values, list):
                ref_values = ref_values[0]
            if isinstance(future_values, list):
                future_values = future_values[0]
            ref_importance = np.abs(ref_values).mean(axis=0)
            forecast_importance = np.abs(future_values).mean(axis=0)
            source = "shap_tree"
        except Exception as exc:
            print(f"SHAP failed for {label}; using native importance instead:", repr(exc))

    if ref_importance is None:
        if hasattr(model, "feature_importances_"):
            ref_importance = np.asarray(model.feature_importances_, dtype=float)
        elif hasattr(model, "coef_"):
            ref_importance = np.abs(np.asarray(model.coef_, dtype=float))
        else:
            ref_importance = np.zeros(X_reference.shape[1])
        forecast_importance = ref_importance.copy()

    ref = pd.DataFrame({
        "feature": X_reference.columns,
        "mean_abs_importance": ref_importance,
        "family": [feature_family(c) for c in X_reference.columns],
        "target": label,
        "scope": "train_reference",
        "importance_source": source,
    }).sort_values("mean_abs_importance", ascending=False)
    future = pd.DataFrame({
        "feature": X_forecast.columns,
        "mean_abs_importance": forecast_importance,
        "family": [feature_family(c) for c in X_forecast.columns],
        "target": label,
        "scope": "forecast_2023_2024",
        "importance_source": source,
    }).sort_values("mean_abs_importance", ascending=False)
    return ref, future


def plot_top_features(importance_df, title, filename, top_n=20):
    top = importance_df.head(top_n).sort_values("mean_abs_importance")
    plt.figure(figsize=(9, max(5, top_n * 0.28)))
    plt.barh(top["feature"], top["mean_abs_importance"])
    plt.title(title)
    plt.xlabel("Mean absolute SHAP value" if "shap" in str(importance_df["importance_source"].iloc[0]) else "Native importance")
    plt.tight_layout()
    out = REPORT_DIR / filename
    plt.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    return out


# Revenue explanation model.
explain_cols = [c for c in feature_cols if c in full_hist.columns and c in test_future.columns]
X_rev, X_rev_future, y_rev, explain_cols = imputed_xy(full_hist, test_future, explain_cols, "Revenue")
rev_explain_model, rev_model_kind = fit_explain_model(X_rev, y_rev)
rev_imp_train, rev_imp_future = compute_importance(rev_explain_model, rev_model_kind, X_rev, X_rev_future, "Revenue")

# COGS ratio explanation model.
ratio_explain_cols = [c for c in ratio_cols if c in ratio_train.columns and c in test_future.columns]
X_ratio, X_ratio_future, y_ratio, ratio_explain_cols = imputed_xy(ratio_train, test_future, ratio_explain_cols, "COGS_ratio_target")
ratio_explain_model, ratio_model_kind = fit_explain_model(X_ratio, y_ratio)
ratio_imp_train, ratio_imp_future = compute_importance(ratio_explain_model, ratio_model_kind, X_ratio, X_ratio_future, "COGS_ratio")

importance_all = pd.concat([rev_imp_train, rev_imp_future, ratio_imp_train, ratio_imp_future], ignore_index=True)
importance_all.to_csv(REPORT_DIR / "03_shap_feature_importance.csv", index=False)

family_importance = (
    importance_all.groupby(["target", "scope", "family", "importance_source"], as_index=False)["mean_abs_importance"]
    .sum()
    .sort_values(["target", "scope", "mean_abs_importance"], ascending=[True, True, False])
)
family_importance.to_csv(REPORT_DIR / "03_shap_family_importance.csv", index=False)

rev_plot = plot_top_features(rev_imp_future, "Revenue forecast drivers, 2023-2024", "03_shap_revenue_top20.png")
ratio_plot = plot_top_features(ratio_imp_future, "COGS ratio forecast drivers, 2023-2024", "03_shap_cogs_ratio_top20.png")

# Business-readable report.
def top_family_lines(df, target, scope, n=8):
    part = df[(df["target"] == target) & (df["scope"] == scope)].head(n)
    return [f"- {row.family}: {row.mean_abs_importance:,.2f}" for row in part.itertuples(index=False)]

report_lines = [
    "# SHAP Explainability Report",
    "",
    "## What This Section Adds",
    "- Trains compact explanation models after final forecast construction.",
    "- Explains Revenue and COGS/Revenue ratio separately.",
    "- Groups raw features into business families so the next improvement direction is visible.",
    "- Uses SHAP TreeExplainer when available; otherwise falls back to native model importance.",
    "",
    f"Revenue explanation model: `{rev_model_kind}`.",
    f"COGS ratio explanation model: `{ratio_model_kind}`.",
    "",
    "## Revenue Forecast Drivers",
    *top_family_lines(family_importance, "Revenue", "forecast_2023_2024"),
    "",
    "## COGS Ratio Forecast Drivers",
    *top_family_lines(family_importance, "COGS_ratio", "forecast_2023_2024"),
    "",
    "## How To Use This For Next Improvements",
    "- If lag momentum dominates, improve recursive future lag generation instead of adding more calendar flags.",
    "- If Tet or fashion seasonality dominates, inspect month/Tet-window residuals and refine seasonal shape.",
    "- If promotion intensity dominates, validate projected 2023-2024 promo recurrence from `promotions.csv`.",
    "- If COGS ratio is driven mostly by calendar rather than promo/category mix, improve margin modeling by category/month.",
]
(REPORT_DIR / "03_shap_business_drivers.md").write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:")
print("-", REPORT_DIR / "03_shap_feature_importance.csv")
print("-", REPORT_DIR / "03_shap_family_importance.csv")
print("-", REPORT_DIR / "03_shap_business_drivers.md")
print("-", rev_plot)
print("-", ratio_plot)

display(Markdown("\n".join(report_lines[:40])))
display(importance_all.head(20))


## Manual Export Cell

In [ ]:
# Safe by default. Change to True only when you want to create outputs/submission.csv yourself.
WRITE_SUBMISSION = True

if WRITE_SUBMISSION:
    submission = final.copy()
    submission["Date"] = submission["Date"].dt.strftime("%Y-%m-%d")
    submission = submission[["Date", "Revenue", "COGS"]]
    out_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(out_path, index=False)
    print("Saved:", out_path)
else:
    print("WRITE_SUBMISSION is False; no submission.csv was created.")
    display(final.head())
